Explore how borders are resampled.

In [ ]:
%matplotlib inline

import healpix_geo
import healpix_plot as hp
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import xarray as xr
from healpix_geo.auto import Grid
from healpix_resample import PSFResampler
from matplotlib.transforms import Bbox
from matplotlib.collections import PolyCollection

from swot_healpix.data import SWOT
from swot_healpix.utils import set_latlon_formatters

In [ ]:
# Load data
def load(cycle: int, track: int, lat: float, side: str = "left") -> xr.Dataset:
    """Load SWOT data."""
    swot = SWOT(cycle=cycle, track=track, version="PGC0")
    swot_file = swot.get_file()
    swath = xr.open_dataset(swot_file, group=side, chunks=-1)

    center_lat = swath.latitude.isel(num_pixels=swath.num_pixels.size // 2).values
    center_lat = np.sort(center_lat)  # just in case, just only modify slighty over land
    i_line = np.searchsorted(center_lat, lat)

    swath = swath.isel(num_lines=slice(i_line - 250, i_line + 250))

    if side == "left":
        swath = swath.isel(num_pixels=slice(None, None, -1))
    if swot.pass_type == "descending":
        swath = swath.isel(num_lines=slice(None, None, -1))

    swath.longitude.load()
    swath.latitude.load()
    swath.ssh_karin_2.load()

    swath = swath.dropna("num_lines", how="all", subset=["longitude"])
    swath = swath.dropna("num_pixels", how="all", subset=["longitude"])

    swath["ssh_karin_2"] -= swath.ssh_karin_2.mean()

    return swath

swath = load(cycle=546, track=19, lat=-46.96, side="left")
swath_lon = swath.longitude.values.ravel()
swath_lat = swath.latitude.values.ravel()
swath_ssh = swath.ssh_karin_2.values.ravel()

In [ ]:
bbox = Bbox.from_extents(140.2, -46.6, 140.6, -46.3)

In [ ]:
fig, ax = plt.subplots(figsize=(3, 2), layout="constrained", dpi=200)

swath.ssh_karin_2.plot.pcolormesh(
    x="longitude",
    y="latitude",
    cmap="Blues_r",
    add_labels=False,
    cbar_kwargs={"pad": 0.02, "label": "[m]"},
)
ax.add_artist(
    plt.Rectangle(bbox.p0, width=bbox.width, height=bbox.height, ec="r", fc="none")
)
ax.set_ylim(None, -46.2)
ax.set_aspect("equal")
set_latlon_formatters(ax)
ax.xaxis.set_major_locator(mtick.MaxNLocator(4))

In [ ]:
level = 12
grid_kw = {"level": level, "indexing_scheme": "nested", "ellipsoid": "WGS84"}
grid =  Grid(**grid_kw)

In [ ]:
cell_ids, _, _ = healpix_geo.auto.zone_coverage(tuple(bbox.extents), grid)

cell_ids_enlarged = (
    healpix_geo.auto.kth_neighbourhood(cell_ids, grid=grid, ring=2)
    .reshape(-1)
    .astype("int64")
)
cell_ids_enlarged = cell_ids_enlarged[cell_ids_enlarged > 0]
cell_ids_enlarged = np.unique(cell_ids_enlarged)

cell_ids_sample = healpix_geo.auto.lonlat_to_healpix(swath_lon, swath_lat, grid=grid)
samples_keep_mask = np.isin(cell_ids_sample, cell_ids_enlarged)

In [ ]:
resampler = PSFResampler(
    lon_deg=swath_lon[samples_keep_mask],
    lat_deg=swath_lat[samples_keep_mask],
    level=grid.level,
    out_cell_ids=cell_ids,
    #sigma_m=275,
)

resampled = resampler.resample(swath_ssh[samples_keep_mask], lam=0.25)
res_cell_ids = resampled.cell_ids
res_cell_data = resampled.cell_data

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2), layout="constrained", dpi=200)

ims = hp.pcolormesh(
    res_cell_ids,
    res_cell_data,
    healpix_grid=grid_kw,
    level_tile=7,
    ax=ax
)
fig.colorbar(ims[0], ax=ax, pad=0.02, label="[m]")

ax.set_aspect("equal")
set_latlon_formatters(ax)
ax.xaxis.set_major_locator(mtick.MaxNLocator(4))

In [ ]:
cell_ids_sample_unique, sample_cell_counts = np.unique(
    cell_ids_sample[samples_keep_mask], return_counts=True
)

fig, ax = plt.subplots(figsize=(4, 2), layout="constrained", dpi=200)

ims = hp.pcolormesh(
    cell_ids_sample_unique,
    sample_cell_counts,
    healpix_grid=grid_kw,
    level_tile=7,
    ax=ax,
)
fig.colorbar(ims[0], ax=ax, pad=0.02, label="sample count")

ax.set_aspect("equal")
set_latlon_formatters(ax)
ax.xaxis.set_major_locator(mtick.MaxNLocator(4))

In [ ]:
threshold = 20
cell_ids_sample_keep_mask = sample_cell_counts > threshold
keep_mask = np.isin(res_cell_ids, cell_ids_sample_unique[cell_ids_sample_keep_mask])
# bad_mask = np.isin(res_cell_ids, cell_ids_sample_unique[~cell_ids_sample_keep_mask])
res_cell_ids_bad = cell_ids_sample_unique[~cell_ids_sample_keep_mask]

# res_cell_ids_bad = res_cell_ids[bad_mask]
res_cell_ids_clean = res_cell_ids[keep_mask]
res_cell_data_clean = res_cell_data[keep_mask]

In [ ]:
fig, axes = plt.subplots(
    1, 2, figsize=(6, 1.8), layout="constrained", dpi=200, sharex=True, sharey=True
)
ax1, ax2 = axes

vmin = min(np.nanmin(res_cell_data), np.nanmin(res_cell_data_clean))
vmax = max(np.nanmax(res_cell_data), np.nanmax(res_cell_data_clean))

hp.pcolormesh(
    res_cell_ids,
    res_cell_data,
    healpix_grid=grid_kw,
    level_tile=7,
    vmin=vmin,
    vmax=vmax,
    ax=ax1,
)

ims = hp.pcolormesh(
    
    # res_cell_ids_clean,
    # res_cell_data_clean,
    healpix_grid=grid_kw,
    level_tile=7,
    # vmin=vmin,
    # vmax=vmax,
    ax=ax2,
)
fig.colorbar(ims[0], ax=axes, pad=0.02, label="[m]")

# bad_vertices = np.stack(healpix_geo.auto.vertices(res_cell_ids_bad, grid=grid), axis=-1)
# ax1.add_collection(PolyCollection(bad_vertices, ec="r", fc="none", lw=0.7))
# ax2.add_collection(PolyCollection(bad_vertices, ec="r", fc="none", lw=0.7))

for ax in axes:
    ax.set_aspect("equal")
    set_latlon_formatters(ax)
    ax.xaxis.set_major_locator(mtick.MaxNLocator(4))